## Day 2 - Part 4: CNN 기초: 이미지를 '보는' 신경망의 비밀

### 개요

Day 2의 앞선 파트들에서 우리는 표(tabular) 형태의 데이터를 다루는 심층 신경망(DNN)을 구축하고, 그 성능을 최적화하는 다양한 기법을 배웠습니다. 

하지만 세상에는 숫자나 표로 정리된 데이터만 있는 것이 아닙니다. 우리 주변을 가득 채운 `이미지` 데이터는 어떻게 처리해야 할까요?

지금까지 배운 DNN, 즉 완전연결(Fully Connected) 신경망에 256x256 픽셀의 컬러 이미지를 넣는다고 상상해 보세요. 

이미지를 단순히 한 줄로 길게 펼치면 입력 노드의 개수가 `256 * 256 * 3 = 196,608`개나 됩니다. 

첫 번째 은닉층에 1000개의 뉴런만 두어도, 입력층과 은닉층 사이의 가중치(파라미터) 개수는 무려 2억 개에 육박합니다\! 이는 엄청난 계산 비용과 과적합 문제를 야기합니다. 

더 심각한 문제는, 이미지의 중요한 `공간적 구조`(spatial structure), 즉 어떤 픽셀이 서로 이웃해 있는지에 대한 정보가 완전히 사라진다는 점입니다.

이러한 한계를 극복하기 위해 등장한 것이 바로 `합성곱 신경망(Convolutional Neural Network, CNN)` 입니다. 

CNN은 인간의 시각 처리 방식에서 영감을 얻어, 이미지의 지역적 특징(local feature)을 효과적으로 추출하고 조합하여 전체를 이해하는, 이미지 처리에 특화된 혁신적인 아키텍처입니다.

이번 파트에서는 DNN의 한계를 넘어 이미지를 '볼' 수 있는 신경망, CNN의 세계로 떠납니다. 

이미지를 분석하는 '돋보기' 역할을 하는 `합성곱(Convolution)` 과 `풀링(Pooling)` 의 원리를 배우고, 이를 조합하여 직접 이미지 분류 모델을 만들어 보겠습니다.

`이번 파트의 학습 목표:`

  * 이미지 데이터 처리에 기존 DNN이 갖는 한계(파라미터 폭증, 공간 정보 손실)를 이해합니다.
  
  * CNN의 핵심 연산인 `합성곱(Convolution)` 의 원리를 이해하고, 필터(커널), 스트라이드(Stride), 패딩(Padding)의 역할을 설명할 수 있습니다.
  * 정보를 압축하고 모델을 강건하게 만드는 `풀링(Pooling)` 연산의 개념과 종류를 이해합니다.
  * 합성곱, 활성화 함수, 풀링, 완전연결층을 조합하여 기본적인 CNN 모델의 전체 구조를 설계하고 PyTorch 코드로 구현할 수 있습니다.
  * `CIFAR-10` 이미지 데이터셋을 사용하여 직접 구축한 CNN 모델을 학습시키고, 그 성능을 평가할 수 있습니다.
  * 학습된 CNN의 필터와 특징 맵(Feature Map)을 시각화하여, 모델이 이미지로부터 무엇을 학습하는지 직관적으로 이해할 수 있습니다.



### 1. 왜 이미지는 DNN으로 어려울까? : 문제의 본질

CNN을 배우기 전, 왜 완전연결 신경망(DNN)이 이미지 처리에 적합하지 않은지 명확히 짚고 넘어가겠습니다.

  * `문제 1: 파라미터의 폭발적인 증가`
    앞서 언급했듯, 이미지를 1차원 벡터로 펼쳐서 DNN에 입력하면 연결 가중치의 수가 기하급수적으로 늘어납니다. 이는 모델을 매우 무겁게 만들고, 학습에 엄청난 시간과 컴퓨팅 자원을 요구하며, 과적합에 극도로 취약하게 만듭니다.

  * `문제 2: 공간적 구조 정보의 손실`
    이미지에서 픽셀의 위치는 매우 중요합니다. 고양이의 눈 옆에는 코가 있고, 자동차 바퀴는 차체 아래에 있습니다. 하지만 이미지를 일렬로 펼치는 순간, 원래 이미지에서 바로 옆에 있던 픽셀과 멀리 떨어져 있던 픽셀은 아무런 차이가 없는 동등한 입력값이 되어버립니다. 엣지, 코너, 질감과 같은 중요한 공간적 특징을 모두 잃게 되는 것입니다.

CNN은 이 두 가지 문제를 '지역적 특징 인식'과 '파라미터 공유'라는 두 가지 천재적인 아이디어로 해결합니다.


### 2. CNN의 핵심 구성 요소 파헤치기

CNN 모델은 여러 종류의 레이어를 블록처럼 쌓아 만듭니다. 가장 핵심적인 블록인 `합성곱 층`과 `풀링 층`에 대해 자세히 알아봅시다.

[구글시트로 이해하기](https://docs.google.com/spreadsheets/d/1wLgJcza-Nj5FdrymS2Y3Qko_pAYS4yvBGgO1jjNCsws/edit?usp=sharing)



#### 2.1. 합성곱(Convolution) 연산: 특징을 포착하는 '필터'

합성곱 연산은 CNN의 심장입니다. 이것은 마치 이미지 위를 미끄러져 다니는 작은 '돋보기' 또는 '필터(Filter)'와 같습니다. 

이 필터는 특정 패턴(예: 수직선, 수평선, 특정 색상 덩어리 등)을 감지하는 역할을 합니다.

  * `필터 (Filter / 커널, Kernel)`: 특정 특징을 감지하기 위한 가중치 행렬입니다. 예를 들어, 3x3 크기의 필터는 이미지의 3x3 영역을 한 번에 보게 됩니다.
  
  * `합성곱 (Convolution)`: 필터를 이미지의 왼쪽 위부터 오른쪽 아래까지 일정 간격으로 이동시키면서, 필터와 겹치는 이미지 영역의 각 픽셀 값을 곱한 후 모두 더하는 연산입니다. 이 결과값 하나가 출력의 한 픽셀이 됩니다.
  * `특징 맵 (Feature Map / 활성 맵, Activation Map)`: 하나의 필터가 이미지 전체를 훑고 지나가면 하나의 출력 이미지가 생성되는데, 이를 특징 맵이라고 합니다. 이는 원본 이미지에서 필터가 감지하려던 특징이 '어디에', '얼마나 강하게' 나타나는지를 보여주는 지도와 같습니다.

CNN에서는 이렇게 각기 다른 특징을 감지하는 여러 개의 필터를 사용하여, 입력 이미지로부터 다양한 측면의 특징 맵들을 동시에 추출합니다.

<img src="https://miro.medium.com/v2/resize:fit:462/format:webp/1*CBY94wikMUCZMB4-Xxs-pw.png">
<img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*XbuW8WuRrAY5pC4t-9DZAQ.jpeg">
<img src="https://miro.medium.com/v2/resize:fit:922/format:webp/1*kYSsNpy0b3fIonQya66VSQ.png">



`코드 실습: `nn.Conv2d`로 합성곱 연산 이해하기`

PyTorch에서는 `nn.Conv2d` 레이어로 합성곱을 구현합니다. `2d`는 2차원 이미지 데이터를 다룬다는 의미입니다.

In [ ]:
import torch
import torch.nn as nn

# (배치 크기, 입력 채널, 높이, 너비) 형태의 더미 이미지 데이터 생성
# 흑백 이미지(채널=1) 1개, 크기는 10x10
dummy_image = torch.randn(1, 1, 10, 10)

# nn.Conv2d 정의 - 합성곱
# in_channels=1: 입력 이미지의 채널 수 (흑백이므로 1)
# out_channels=8: 사용할 필터의 개수. 즉, 출력될 특징 맵의 개수
# kernel_size=3: 필터(커널)의 크기 (3x3)
conv_layer = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3)

# 합성곱 연산 수행
feature_maps = conv_layer(dummy_image)

print("Original Image Shape:", dummy_image.shape)
print("Feature Maps Shape:", feature_maps.shape)

Original Image Shape: torch.Size([1, 1, 10, 10])
Feature Maps Shape: torch.Size([1, 8, 8, 8])


위 코드를 실행하면 `(1, 8, 8, 8)` 형태의 출력을 볼 수 있습니다. 10x10 이미지에 3x3 필터를 적용하면 왜 8x8 특징 맵이 나올까요? 이는 필터가 이미지 가장자리를 벗어날 수 없기 때문입니다. 이 출력 크기를 조절하는 것이 바로 스트라이드와 패딩입니다.

#### 2.2. 스트라이드(Stride)와 패딩(Padding)

  <img src="https://miro.medium.com/v2/resize:fit:1390/format:webp/1*nGHLq1hx0gt02OK4l8WmRg.png">
  <div>
  <img src="https://miro.medium.com/v2/resize:fit:826/format:webp/1*4yv0yIH0nVhSOv3AkLUIiw.png">
  <img src="https://miro.medium.com/v2/resize:fit:536/format:webp/1*MrGSULUtkXc0Ou07QouV8A.gif">
  </div>

  * `스트라이드 (Stride)`: 필터가 이미지 위를 이동하는 '보폭'의 크기입니다. `stride=1`이면 한 픽셀씩, `stride=2`이면 두 픽셀씩 건너뛰며 이동합니다. 스트라이드가 커지면 출력 특징 맵의 크기는 작아지고, 계산량도 줄어듭니다.

  * `패딩 (Padding)`: 합성곱 연산을 하기 전에 입력 이미지의 가장자리를 특정 값(보통 0)으로 둘러싸는 것입니다.

      * `목적 1 (크기 보존)`: 패딩을 사용하면 합성곱 연산 후에도 출력 특징 맵의 크기가 입력과 동일하게 유지되도록 조절할 수 있습니다. (`padding = (kernel_size - 1) / 2` 일 때)
      * `목적 2 (정보 손실 방지)`: 패딩이 없으면 이미지의 모서리나 경계 부분의 픽셀들은 필터와 적게 겹치게 되어 정보가 소실될 수 있습니다. 패딩은 이 문제를 완화합니다.
  
  


`코드 실습: 스트라이드와 패딩 적용하기`

In [2]:
# Stride=2, Padding=1 을 적용한 합성곱 레이어
conv_layer_sp = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=2, padding=1)

feature_maps_sp = conv_layer_sp(dummy_image)

print("Original Image Shape:", dummy_image.shape)
print("Feature Maps Shape (with Stride & Padding):", feature_maps_sp.shape)

Original Image Shape: torch.Size([1, 1, 10, 10])
Feature Maps Shape (with Stride & Padding): torch.Size([1, 8, 5, 5])


`padding=1`을 통해 10x10 이미지가 12x12로 확장되고, 여기에 3x3 필터를 `stride=2`로 적용하면 출력 크기는 5x5가 됩니다. 이처럼 두 파라미터로 출력 크기를 자유롭게 조절할 수 있습니다.

#### 2.3. 풀링(Pooling) 연산: 정보 압축과 일반화

합성곱 층이 특징을 추출했다면, `풀링(Pooling)` 층은 이 특징 맵에서 중요하지 않은 정보를 버리고 핵심 정보만 남겨 `정보를 압축(down-sampling)`하는 역할을 합니다.

  * `주요 목적`:

    1.  `계산 효율성`: 특징 맵의 크기를 줄여 다음 레이어의 파라미터 수를 줄이고 계산량을 감소시킵니다.
    2.  `과적합 억제`: 미세한 변화는 무시하고 핵심 특징만 남기므로 모델의 일반화 성능을 높입니다.
    3.  `이동 불변성(Translation Invariance)`: 이미지에서 특징의 위치가 약간 변하더라도 풀링을 거치면 결과가 크게 달라지지 않는 효과를 줍니다.

  * `종류`:

      * `최대 풀링 (Max Pooling)`: 풀링 영역 내에서 가장 큰(가장 강하게 활성화된) 값만 남깁니다. 특징의 존재 여부를 포착하는 데 효과적이라 가장 널리 사용됩니다.
      * `평균 풀링 (Average Pooling)`: 풀링 영역 내의 모든 값의 평균을 취합니다.

`코드 실습: `nn.MaxPool2d`로 최대 풀링 적용하기`

풀링은 보통 활성화 함수(예: ReLU)를 거친 특징 맵에 적용됩니다.

In [3]:
# 8개의 8x8 특징 맵이 있다고 가정
feature_maps_from_conv = torch.randn(1, 8, 8, 8)

# nn.MaxPool2d 정의
# kernel_size=2: 풀링을 적용할 영역의 크기 (2x2)
# stride=2: 풀링 윈도우의 이동 보폭
max_pool_layer = nn.MaxPool2d(kernel_size=2, stride=2) # stride - *2 하는 것처럼 보폭을 늘려줌

pooled_maps = max_pool_layer(feature_maps_from_conv)

print("Before Pooling Shape:", feature_maps_from_conv.shape)
print("After Pooling Shape:", pooled_maps.shape)

Before Pooling Shape: torch.Size([1, 8, 8, 8])
After Pooling Shape: torch.Size([1, 8, 4, 4])


2x2 최대 풀링을 `stride=2`로 적용하면 특징 맵의 가로, 세로 크기가 정확히 절반으로 줄어드는 것을 확인할 수 있습니다.

#### 2.4. 전체 구조 조립하기: Conv → ReLU → Pool → FC

이제 배운 요소들을 조립하여 전형적인 CNN 분류 모델의 구조를 완성해 봅시다.

1.  `입력 이미지`
2.  `[합성곱(Conv) → 활성화(ReLU) → 풀링(Pool)] 블록`: 이 블록을 여러 번 반복해서 쌓습니다. 초기 층에서는 엣지, 질감 같은 저수준(low-level) 특징을, 깊은 층으로 갈수록 눈, 코, 바퀴 같은 고수준(high-level)의 복잡한 특징을 학습합니다.
3.  `평탄화 (Flatten)`: 여러 층의 합성곱과 풀링을 거쳐 최종적으로 얻어진 다차원 특징 맵을 1차원 벡터로 길게 펼칩니다. 이는 DNN(완전연결층)에 입력하기 위한 과정입니다.
4.  `완전연결층 (Fully Connected Layer)`: 평탄화된 특징 벡터를 입력으로 받아, 최종적으로 이미지가 어떤 클래스에 속할지 분류하는 역할을 합니다. 이 부분은 우리가 Day 2 Part 1에서 배운 DNN과 동일합니다.

### 3\. 종합 실습: CIFAR-10 데이터셋으로 첫 CNN 모델 구축하기

이제 이론을 바탕으로, 10가지 종류의 컬러 이미지를 담고 있는 `CIFAR-10` 데이터셋을 분류하는 CNN 모델을 직접 만들어 보겠습니다.

#### 3.1. 데이터 준비 및 전처리

CIFAR-10은 32x32 크기의 컬러(3채널) 이미지 데이터셋입니다. `torchvision`을 사용하면 쉽게 데이터를 불러오고, `transforms`를 통해 텐서 변환 및 정규화를 수행할 수 있습니다.

In [4]:
!pip install torchvision

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import plotly.express as px
import numpy as np

# 데이터 전처리 정의
# ToTensor(): PIL 이미지를 PyTorch 텐서로 변환 (값 범위를 [0, 1]로 조정)
# Normalize(): 텐서 이미지의 각 채널을 (평균, 표준편차)로 정규화
# 이 값들은 CIFAR-10 데이터셋의 R, G, B 채널별 평균과 표준편차임
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# CIFAR-10 훈련/테스트 데이터셋 다운로드
path = "../datasets/dl/cifar10/" # 로컬/도커 환경에 따라 알맞게 경로 변경 할것.
train_dataset = torchvision.datasets.CIFAR10(root=path, train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=path, train=False, download=True, transform=transform)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 클래스 이름 정의
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:23<00:00, 7.32MB/s] 


#### 3.2. CNN 모델 정의하기

2개의 합성곱-풀링 블록과 2개의 완전연결층으로 구성된 간단한 CNN 모델을 설계해 보겠습니다. 각 단계에서 데이터의 형태(shape)가 어떻게 변하는지 주석으로 확인하는 것이 중요합니다.

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 입력: (N, 3, 32, 32)
        self.conv_block1 = nn.Sequential(
            # 3채널 입력을 16채널 출력으로, 5x5 커널, 패딩 2
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2), # -> (N, 16, 32, 32)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # -> (N, 16, 16, 16)
        )
        self.conv_block2 = nn.Sequential(
            # 16채널 입력을 32채널 출력으로, 5x5 커널, 패딩 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, padding=2), # -> (N, 32, 16, 16)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # -> (N, 32, 8, 8)
        )
        # 평탄화 이후 입력될 완전연결층
        # 32개의 8x8 특징 맵을 펼치면 -> 32 * 8 * 8 = 2048
        self.fc_block = nn.Sequential(
            nn.Linear(32 * 8 * 8, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10) # 10개 클래스에 대한 출력
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        # .view()를 사용하여 평탄화 (배치 차원은 유지)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

# 모델 인스턴스 생성 및 device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv_block1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_block): Sequential(
    (0): Linear(in_features=2048, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=10, bias=True)
  )
)


#### 3.3. 모델 학습 및 평가

이제 손실 함수와 옵티마이저를 설정하고, 훈련 루프를 만들어 모델을 학습시킵니다. 이 과정은 DNN을 학습시킬 때와 거의 동일합니다.

In [7]:
# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모델 학습 함수
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.3f}')

    print('Finished Training')

# 모델 학습 실행
train_model(model, train_loader, criterion, optimizer, num_epochs=10)

# 모델 평가
model.eval() # 평가 모드
correct = 0
total = 0
with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'네트워크가 10000개 테스트 이미지에서 달성한 정확도: {100 * correct / total:.2f} %')

Epoch 1, Loss: 1.482
Epoch 2, Loss: 1.102
Epoch 3, Loss: 0.935
Epoch 4, Loss: 0.817
Epoch 5, Loss: 0.727
Epoch 6, Loss: 0.650
Epoch 7, Loss: 0.579
Epoch 8, Loss: 0.517
Epoch 9, Loss: 0.460
Epoch 10, Loss: 0.406
Finished Training
네트워크가 10000개 테스트 이미지에서 달성한 정확도: 70.20 %


#### 3.4. CNN 내부 들여다보기: 특징 맵 시각화

CNN이 정말 이미지의 특징을 학습하는지 눈으로 직접 확인해 봅시다. 테스트 이미지 하나를 모델의 첫 번째 합성곱 블록에 통과시킨 후, 생성된 16개의 특징 맵을 시각화해 보겠습니다.

In [8]:
# 테스트 데이터에서 이미지 하나 가져오기
dataiter = iter(test_loader)
images, labels = next(dataiter)

# 첫 번째 이미지와 레이블
img = images[0]
label = labels[0]

# 이미지 시각화를 위해 정규화 되돌리기
img_for_show = img / 2 + 0.5
npimg = img_for_show.numpy()
px.imshow(np.transpose(npimg, (1, 2, 0)), title=f"Original Image: {classes[label]}").show()


# 첫 번째 합성곱 블록을 통과한 후의 특징 맵 추출
model.eval()
with torch.no_grad():
    # 모델 입력에 맞게 배치 차원 추가
    feature_maps = model.conv_block1(img.unsqueeze(0).to(device))

feature_maps = feature_maps.squeeze(0).cpu().numpy()

# 16개 특징 맵 시각화
fig = px.imshow(feature_maps, facet_col=0, facet_col_wrap=4,
                labels={'facet_col':'Feature Map Index'},
                color_continuous_scale='viridis')
fig.update_layout(title='Feature Maps after First Conv-Pool Block')
fig.show()

시각화된 특징 맵들을 보면, 어떤 맵은 수직선, 어떤 맵은 배경, 또 다른 맵은 객체의 윤곽선을 강조하는 등 각 필터가 이미지의 서로 다른 측면을 포착했음을 직관적으로 알 수 있습니다. 이것이 바로 CNN이 이미지를 '이해'하는 방식입니다.